In [ ]:
# Install the Isaacus SDK and Isaacus SageMaker Python integration.
!pip install isaacus isaacus-sagemaker

# Install optional dependencies necessary for this example (`httpx` and `numpy`).
!pip install httpx numpy

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.3/136.3 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.8/365.8 kB 160.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 168.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 69.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver 

In [ ]:
from isaacus import Isaacus, AsyncIsaacus
from isaacus_sagemaker import IsaacusSageMakerRuntimeHTTPClient, AsyncIsaacusSageMakerRuntimeHTTPClient, \
     IsaacusSageMakerRuntimeEndpoint

# NOTE This assumes you have already purchased Kanon 2 Embedder and set up an endpoint.
endpoints = [
    IsaacusSageMakerRuntimeEndpoint(
        name="my-sagemaker-endpoint",
        # region="us-west-2", # Optional, defaults to the client or AWS SDK default region
        # profile="my-aws-profile", # Optional, defaults to the client or AWS SDK default profile
        # models=["kanon-2-embedder"], # Optional, models supported by this endpoint,
        #                              # defaults to all models
    )
]

client = Isaacus(
    http_client=IsaacusSageMakerRuntimeHTTPClient(
        endpoints=endpoints,
        # region="us-west-2", # Optional, defaults to AWS SDK default region
        # profile="my-aws-profile", # Optional, defaults to AWS SDK default profile
        # boto_session_kwargs={"aws_access_key_id": "...",}, # Optional, additional boto3 session kwargs
        # **{}, # Optional, additional httpx.Client kwargs
    )
)

# For asynchronous usage:
aclient = AsyncIsaacus(
    http_client=AsyncIsaacusSageMakerRuntimeHTTPClient(
        endpoints=endpoints,
        # region="us-west-2", # Optional, defaults to AWS SDK default region
        # profile="my-aws-profile", # Optional, defaults to AWS SDK default profile
        # boto_session_kwargs={"aws_access_key_id": "...",}, # Optional, additional boto3 session kwargs
        # **{}, # Optional, additional httpx.AsyncClient kwargs
    )
)

In [ ]:
import httpx
import numpy as np

# Download the GitHub terms of service as an example.
tos = httpx.get("https://examples.isaacus.com/github-tos.md").text

# Embed the terms of service.
document_response = client.embeddings.create(
    model="kanon-2-embedder",
    texts=tos, # You can pass a single text or a list of texts here.
    task="retrieval/document",
    # dimensions=1792, # You may optionally wish to specify a lower dimension.
)

# Embed our search queries.
query_responses = client.embeddings.create(
    model="kanon-2-embedder",
    texts=[
        "What are GitHub's billing policies?", # This is a relevant query.
        "What are Microsoft's billing policies?", # This is an irrelevant query.
    ],
    task="retrieval/query",
    # dimensions=1792, # You may optionally wish to specify a lower dimension.
)

# Unpack the embeddings.
document_embedding = document_response.embeddings[0].embedding

query_embeddings = query_responses.embeddings
relevant_query_embedding = query_embeddings[0].embedding
irrelevant_query_embedding = query_embeddings[1].embedding

# Compute the similarity between the queries and the document.
relevant_similarity = np.dot(relevant_query_embedding, document_embedding)
irrelevant_similarity = np.dot(irrelevant_query_embedding, document_embedding)

# Log the results.
print(f"Similarity of relevant query to the document: {relevant_similarity * 100:.2f}")
print(f"Similarity of irrelevant query to the document: {irrelevant_similarity * 100:.2f}")